## **Asistente de Documentos con Gemini y Gradio**

#### Integrantes de Equipo:
- Juan José Escobar
- Juan David Salas

### **Paso 0: Instalación y configuración**

In [1]:
%pip install pypdf gradio google-genai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


### **Paso 1: Extracción de texto del PDF**

In [2]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        texto_completo = ""
        for pagina in reader.pages:
            texto_completo += pagina.extract_text()
        return texto_completo
    except FileNotFoundError:
        print(f"No se encontró el archivo: {pdf_path}")
        print(f"Verifica que el PDF esté en la misma carpeta que el notebook.")
        return None
    except Exception as e:
        print(f"Error al leer el PDF: {e}")
        return None

PDF_PATH = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(PDF_PATH)

if document_text:
    print(f"PDF leído correctamente")
    print(f"Páginas extraídas: {len(PdfReader(PDF_PATH).pages)}")
    print(f"Caracteres totales: {len(document_text):,}")
    print(f"\nPrimeros 500 caracteres:\n{'-'*40}\n{document_text[:500]}")

PDF leído correctamente
Páginas extraídas: 15
Caracteres totales: 39,615

Primeros 500 caracteres:
----------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


### **Paso 2: Inicialización del cliente de Gemini**

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    print("No se encontró GEMINI_API_KEY en el archivo .env")
    print("   Asegúrate de que el archivo .env existe y contiene:")
    print('   GEMINI_API_KEY="AIza..."')
else:
    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        MODELO = "gemini-2.5-flash-lite"
        print(f"Cliente inicializado correctamente")
        print(f"   Modelo: {MODELO}")
        print(f"   API Key: {GEMINI_API_KEY[:4]}...{GEMINI_API_KEY[-1:]}")
    except Exception as e:
        print(f"Error al inicializar el cliente: {e}")

Cliente inicializado correctamente
   Modelo: gemini-2.5-flash-lite
   API Key: AIza...A


### **Paso 3: Función de chat con contexto del documento**

In [4]:
def build_system_prompt(document_text):
    prompt = f"""Eres un asistente experto dedicado exclusivamente a responder preguntas basadas en el documento proporcionado.
    
Tus instrucciones estrictas son:
1. Responde ÚNICAMENTE utilizando la información contenida en el texto del documento.
2. Si el usuario hace una pregunta cuya respuesta no se encuentra en el documento, debes responder cortésmente: "Lo siento, pero la información para responder a esta pregunta no se encuentra en el documento proporcionado."
3. No uses tu conocimiento externo ni asumas información que no esté explícita en el texto.
--- INICIO DEL DOCUMENTO ---
{document_text}
--- FIN DEL DOCUMENTO ---
"""
    return prompt

def _extract_text(msg):
    if isinstance(msg, str):
        return msg
    elif isinstance(msg, list):
        text = ""
        for m in msg:
            if isinstance(m, dict) and "text" in m:
                text += m["text"]
            elif isinstance(m, str):
                text += m
        return text
    elif isinstance(msg, dict) and "text" in msg:
        return msg["text"]
    return str(msg)

def chat_con_documento(message, history, document_text=document_text):
    try:
        # 1. Construir el system prompt
        system_instruction = build_system_prompt(document_text)
        
        # 2. Formatear el historial para Gemini
        contents = []
        for item in history:
            if isinstance(item, dict):
                role = "model" if item.get("role") == "assistant" else "user"
                content = _extract_text(item.get("content", ""))
                contents.append(
                    types.Content(role=role, parts=[types.Part.from_text(text=content)])
                )
            else:
                user_msg, bot_msg = item
                user_text = _extract_text(user_msg)
                bot_text = _extract_text(bot_msg)
                
                if user_text:
                    contents.append(
                        types.Content(role="user", parts=[types.Part.from_text(text=user_text)])
                    )
                if bot_text:
                    contents.append(
                        types.Content(role="model", parts=[types.Part.from_text(text=bot_text)])
                    )
        
        # 3. Agregar el mensaje actual del usuario
        current_message = _extract_text(message)
        contents.append(
            types.Content(role="user", parts=[types.Part.from_text(text=current_message)])
        )
        
        # 4. Configurar el system instruction y realizar la petición en streaming
        config = types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2
        )
        
        # --- NUEVO: Conteo de tokens dinámico ---
        token_info = ""
        try:
            # Construir contents completo incluyendo el system prompt para el conteo
            contents_para_contar = [
                types.Content(role="user", parts=[types.Part.from_text(text=system_instruction)])
            ] + contents
            
            response_tokens = client.models.count_tokens(
                model=MODELO,
                contents=contents_para_contar
            )
            tot = response_tokens.total_tokens
            pct = (tot / 1_000_000) * 100
            token_info = f"\n\n---\n*📊 Tokens en este turno: {tot:,} ({pct:.4f}%)*"
        except Exception as e:
            print("Error contando tokens:", e)
            token_info = ""
        # ----------------------------------------
        
        response_stream = client.models.generate_content_stream(
            model=MODELO,
            contents=contents,
            config=config
        )
        
        # 5. Generar la respuesta en trozos
        respuesta_parcial = ""
        for chunk in response_stream:
            if chunk.text:
                respuesta_parcial += chunk.text
                yield respuesta_parcial
                
        # 6. Al finalizar, adjuntamos la información de tokens
        if token_info:
            respuesta_parcial += token_info
            yield respuesta_parcial
            
    except Exception as e:
        import traceback
        print(f"ERROR: {e}")
        yield f"Ocurrió un error: {str(e)}"


### **Bonus A: Indicador de Tokens**
Calculamos cuántos tokens consume nuestro system prompt (que incluye el texto completo del paper) y qué porcentaje representa del límite de 1,000,000 de tokens de Gemini 2.5 Flash.

In [5]:
# Construimos el prompt completo para medirlo
system_instruction = build_system_prompt(document_text)

# Usamos la API para contar los tokens exactamente como los ve el modelo
try:
    response_tokens = client.models.count_tokens(
        model=MODELO,
        contents=[
            types.Content(role="user", parts=[types.Part.from_text(text=system_instruction)])
        ]
    )
    
    total_tokens = response_tokens.total_tokens
    limite_tokens = 1_000_000
    porcentaje = (total_tokens / limite_tokens) * 100
    
    print(f"📊 Análisis de Tokens del System Prompt:")
    print(f"   - Tokens utilizados: {total_tokens:,}")
    print(f"   - Límite del modelo ({MODELO}): {limite_tokens:,}")
    print(f"   - Porcentaje del límite utilizado: {porcentaje:.4f}%")
    
    # Si quieres, también podemos guardar esto para mostrarlo en Gradio luego
    info_tokens = f"Tokens usados: {total_tokens:,} ({porcentaje:.2f}% de 1M)"
except Exception as e:
    print(f"Error al contar tokens: {e}")


📊 Análisis de Tokens del System Prompt:
   - Tokens utilizados: 10,951
   - Límite del modelo (gemini-2.5-flash-lite): 1,000,000
   - Porcentaje del límite utilizado: 1.0951%


### **Paso 4: Interfaz Gradio**

In [6]:
import gradio as gr
# 1. Asegurar que tenemos el texto (ya se extrajo en el Paso 1, pero podemos validarlo)
if 'document_text' not in locals() or not document_text:
    print("El texto del documento no está disponible. Asegúrate de ejecutar el Paso 1 primero.")
else:
    print(f"Preparando interfaz con documento de {len(document_text):,} caracteres.")
    # 2. Configurar la interfaz de Chat de Gradio
    demo = gr.ChatInterface(
        fn=chat_con_documento,
        title=" Asistente Experto: Attention Is All You Need",
        description=f"Pregúntame cualquier cosa sobre el famoso paper 'Attention Is All You Need'. Solo responderé basándome en el contenido del documento.\n\n**{info_tokens if 'info_tokens' in globals() else ''}**",

        
        # 3. Inputs adicionales ocultos para pasar el contexto
        additional_inputs=[
            gr.Textbox(value=document_text, visible=False, label="Contexto del Documento")
        ],
        
        # 4. Ejemplos para guiar al usuario
        examples=[
            ["¿Qué es el mecanismo de atención?"],
            ["¿Por qué los autores dicen que no se necesitan RNNs o CNNs?"],
            ["¿Cuáles son los componentes principales de la arquitectura Transformer?"],
            ["Resume los resultados experimentales en tareas de traducción."]
        ],
        
    )
    # 5. Lanzar la aplicación
    # Se añade share=True opcionalmente si se quiere probar temporalmente desde fuera
    demo.launch(server_name="0.0.0.0", server_port=8081
                )


C:\Users\jjesc\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Preparando interfaz con documento de 39,615 caracteres.
* Running on local URL:  http://0.0.0.0:8081
* To create a public link, set `share=True` in `launch()`.


### **Paso 5: Prueba y reflexión**

**Preguntas de Prueba**

---

**1. ¿Cuál es la arquitectura principal propuesta en el paper?**

> La arquitectura principal propuesta en el paper es el **Transformer**, que se basa únicamente en mecanismos de atención, prescindiendo por completo de la recurrencia y las convoluciones.

---

**2. ¿Qué es el mecanismo de atención?**

> Un mecanismo de atención se puede describir como un mapeo de una consulta y un conjunto de pares clave-valor a una salida, donde la consulta, las claves, los valores y la salida son todos vectores. La salida se calcula como una suma ponderada de los valores, donde el peso asignado a cada valor se calcula mediante una función de compatibilidad de la consulta con la clave correspondiente.

---

**3. ¿Cuántas capas tiene el encoder del modelo base?**

> El encoder está compuesto por una pila de **N = 6 capas idénticas**.

---

**4. ¿Quiénes son los autores del paper?**

> Los autores del paper son Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser e Illia Polosukhin.

---

**5. ¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?**

> El modelo Transformer (big) logra un puntaje **BLEU de 28.4** en la tarea de traducción Inglés-Alemán WMT 2014.

---

**6. ¿Qué es GPT-4?** *(pregunta trampa - información fuera del documento)*

> Lo siento, pero la información para responder a esta pregunta no se encuentra en el documento proporcionado.

**Reflexión**

---

**¿Cuál es la limitación principal de este enfoque?**

> La limitación principal es el **tamaño del contexto**. Este enfoque inyecta el documento completo en el system prompt en cada turno de conversación, lo que funciona bien para documentos cortos como este paper (39,615 caracteres, ~10,000 tokens). Sin embargo, para documentos más largos, como un libro, un contrato extenso, una base de conocimiento corporativa, el texto podría superar el límite del modelo o degradar la calidad de las respuestas: los modelos tienden a "perder atención" en fragmentos lejanos del contexto cuando este es muy extenso. Además, enviar el documento completo en cada petición aumenta el costo computacional y la latencia innecesariamente cuando la pregunta solo requiere una pequeña parte del texto.

---

**¿Por qué existe RAG?**

> RAG (*Retrieval-Augmented Generation*) existe precisamente para resolver la limitación anterior. En lugar de inyectar el documento completo, RAG divide el documento en fragmentos pequeños (*chunks*), los indexa en una base de datos vectorial, y en cada pregunta recupera únicamente los fragmentos más relevantes para esa consulta específica. Así el modelo recibe solo el contexto necesario, no todo el documento. Esto permite escalar a documentos arbitrariamente largos, reduce costos y mejora la precisión de las respuestas al eliminar ruido irrelevante del contexto.

---

**¿Qué información podría "filtrarse" aunque el system prompt diga que no?**

> El modelo podría filtrar conocimiento previo de su entrenamiento, y este paper es un ejemplo perfecto: "Attention is All You Need" es uno de los papers más citados de la historia de la IA, por lo que Gemini casi con certeza fue entrenado con él. Aunque el system prompt instruya a responder *solo* con el documento, no es posible "borrar" lo que el modelo ya sabe; las instrucciones reducen la probabilidad de que use conocimiento externo, pero no lo eliminan. Esto quedó evidenciado en la pregunta trampa sobre GPT-4: el modelo respetó la restricción, pero si se le hubiera preguntado algo sobre Transformers con una formulación ambigua, podría haber complementado la respuesta con información de su entrenamiento sin que el usuario lo notara. Para verificar si el modelo está respondiendo desde el documento o desde su memoria, habría que hacerle preguntas sobre información deliberadamente incorrecta o ficticia insertada en el texto, si la responde "correctamente" según el documento aunque sea falsa, está leyendo el documento; si la corrige, está usando conocimiento externo.